# Notebook for batch processing (Running Gears)
Niall Bourke  & Hajer Karoui

Last updated: 04/02/2025   

**Summary**
This script automates the process of searching for specific MRI files in a project, preparing them for analysis, and submitting them to run using a gear. It includes error handling and organizes files into the correct format for submission, making it an efficient way to manage MRI data processing.

This Python code is designed to loop through all the subjects and sessions in a project, and for each session, it will:
- Iterate through Acquisitions: The code iterates through all the acquisitions in each session, looking for specific Nifti files that match certain criteria (e.g., 'T2', 'AXI', 'COR', 'SAG', and excluding 'Segmentation' and 'Align'). These files are then stored in a dictionary called inputs.  
- Set Target Template: The code sets the target_template variable to 'None', which is used as the default configuration for the analysis.   
- Run Analysis: The code then attempts to run an analysis using the gear.run() function, passing in the analysis_label, inputs, destination (the current session), tags, and a configuration dictionary that includes the target_template and prefix values.  
- Error Handling: If there is an error running the analysis, the code will print a warning message.  

\* This example is running the MRR (Multi Resolution Reconstruction) gear, which takes AXI, COR, SAG acquisitions from Hyperfine Swoop as input to create an isotropic reconstruction   

**Notebook environment setup**
- Load required python packages
- Check connection to project through Flywheel SDK
- Set Gear options

---

### Setup options

In [ ]:
import os
from datetime import datetime
import pytz
from IPython.display import display
import re
import pandas as pd
import math

In [ ]:
import os
import subprocess

# Source zshrc and get the env var you want
def get_env_from_zshrc(var_name):
    command = f"source ~/.zshrc && echo ${var_name}"
    result = subprocess.run(['zsh', '-c', command], stdout=subprocess.PIPE, text=True)
    return result.stdout.strip()

api_key = get_env_from_zshrc('FW_CLI_API_KEY')


from dotenv import load_dotenv
import flywheel
# Connect to your Flywheel instance

fw = flywheel.Client(api_key=api_key)
display(f"User: {fw.get_current_user().firstname} {fw.get_current_user().lastname}")
fw_project = fw.projects.find_first('label=DEMO-Zambia')

In [ ]:
# import glob
# import os

# folder_path = "/Users/Hajer/unity/analysis/derivatives"
# csv_files = glob.glob(os.path.join(folder_path, "*.csv"))

# print("Found CSV files:")
# files = []
# for file in csv_files:
#     #print(file)
#     files.append(file)

# group_name = "global_map" 
# group = fw.lookup(group_name)
# projects = group.projects()
# project_csv_mapping={}

# # Loop over all projects
# for project in projects:
#     csv = next((file for file in files if project.label.replace(" ","_") in file), None)

#     if csv:
#         print(f"Project {project.label} has file {csv}")
#         project_csv_mapping[project.label] = csv
#     else:
#         print(f"⚠️  No CSV found for project {project.label}")


In [ ]:
display(f"User: {fw.get_current_user().id}")

project=fw_project
project = project.reload()

In [ ]:
gear =  fw.lookup('gears/mrr')
analysis_tag = 'mrr-axireg'
 # Initialize gear_job_list
job_list = list()

In [ ]:
subjects =[]
sessions = []
for session in project.sessions():
    session = session.reload()
    mrr_asys = [asys for asys in session.analyses if asys.gear_info is not None and asys.gear_info.get('name') == "mrr" and asys.job.get('state') == 'complete']
    if not mrr_asys:
        sessions.append(session)
        

In [ ]:
# testing on a single subject


# Loop through all the subjects in the project
for session in sessions: 
    
        session = session.reload()
        print("Parsing... ", session.subject.label, session.label)
        
        inputs = {}
        EXCLUDE_PATTERNS = ['Segmentation', 'Align', 'Mapping']
        INCLUDE_PATTERN = 'T2'
        PLANE_TYPES = ['AXI', 'COR', 'SAG']

    # Look at every acquisition in the session
        for acquisition in session.acquisitions.iter():
            acquisition = acquisition.reload()
            for file in acquisition.files:
            # We only want anatomical Nifti's
                if file.type == 'nifti' and INCLUDE_PATTERN in file.name:
                    if all(pattern not in file.name for pattern in EXCLUDE_PATTERNS):
                        for plane in PLANE_TYPES:
                            if plane in file.name:
                                input_label = plane.lower()
                                inputs[input_label] = file
                                print("inputs: ", file.name)
                                break

        # Set to 'None' in order to run axi registration by default
            target_template = 'None'

            try:
            # The destination for this analysis will be on the session
                dest = session
                time_fmt = '%d-%m-%Y_%H-%M-%S'
                analysis_label = f'{analysis_tag}_{datetime.now().strftime(time_fmt)}'
                job_id = gear.run(
                    analysis_label=analysis_label,
                    inputs=inputs,
                    destination=dest,
                    tags=[''],
                    config={
                        "target_template": target_template,
                        "prefix": analysis_tag,
                    }
                )
                job_list.append(job_id)
                print("Submitting Job: Check Jobs Log", dest.label)
            except Exception as e:
                print(f"WARNING: Job cannot be sent for {dest.label}. Error: {e}")
            

---

# Code Walkthrough

### 1. Loop Through All Subjects and Sessions

**Loop through all the subjects in the project**

```
for subject in project.subjects.iter():  
    for session in subject.sessions.iter():  
        session = session.reload()  
        print("parsing... ", subject.label, session.label)  

```

- **project.subjects.iter():** This iterates over each subject in the project.  
- **subject.sessions.iter():** This iterates over each session for the current subject.  
- **session.reload():** Reloads the session data to ensure it is up-to-date.  
- **print("parsing... ", subject.label, session.label):** Prints the subject and session labels to show which session is currently being processed.  

### 2. Prepare to Identify Files

`inputs = {}`

- **inputs:* An empty dictionary created to store specific MRI data files that meet our criteria.
- Resetting this distionary to empty is important to make sure a previous subjects files does not exist in it, if the current subject has no matching acquisitions



### 3. Loop Through Acquisitions in the Session

```
# Look at every acquisition in the session
for acq in session.acquisitions.iter():
    acq = acq.reload()
    for file_obj in acq.files:
        # We only want anatomical Nifti's
```

**session.acquisitions.iter():** Loops over all the acquisitions (collections of related images) within the session.  
**acq.reload():** Reloads acquisition data to ensure it is current.  
**file_obj:** Represents each file in the acquisition. The script checks if these files are anatomical MRI images in the NIfTI format.  


### 4. Identify Specific NIfTI Files
The script searches for files matching certain criteria:

``` 
if file_obj.type == 'nifti' and 'T2' in file_obj.name and 'AXI' in file_obj.name and 'Segmentation' not in file_obj.name and 'Align' not in file_obj.name:
    input_label = 'axi'
    inputs[input_label] = file_obj
    print("inputs: ", file_obj.name)
```

**file_obj.type == 'nifti':** Ensures the file is of type 'nifti', a common format for storing MRI data.  
**'T2' in file_obj.name:** Checks if the file name contains 'T2', indicating a T2-weighted MRI.  
**'AXI' in file_obj.name:** Searches for axial plane images ('AXI' indicates axial).  
**Segmentation and Align Checks:** Ensures the file name does not contain 'Segmentation' or 'Align', filtering out irrelevant files.  
**inputs[input_label] = file_obj:** Stores the matching file in the inputs dictionary, using 'axi', 'cor', or 'sag' as keys.  

The same logic applies to identifying:  
**'COR':** Coronal plane images.  
**'SAG':** Sagittal plane images.  


### 5. Run the Analysis

```
try:
    dest = session
    time_fmt = '%d-%m-%Y_%H-%M-%S'
    analysis_label = f'mrr_axireg_{datetime.now().strftime(time_fmt)}'
    job_id = gear.run(analysis_label=analysis_label, inputs=inputs, destination=dest, tags=['BIDS'], config={
        "target_template": target_template,
        "prefix": "mrr-axireg",
    })
    job_list.append(job_id)
    print("Submitting Job: Check Jobs Log", dest.label)
except:
    print("WARNING: Job cannot be sent.. ", dest.label)
    
```

**try-except Block:** Attempts to submit a job for analysis and handles errors gracefully.  
**dest = session:** Sets the session as the destination where the analysis results will be stored.  
**time_fmt:** Defines a date-time format for naming the analysis.  
**analysis_label:** Creates a unique label for the analysis using the current date and time.   
**gear.run(...):** Submits the job for analysis with:  
- **analysis_label:** The label for this analysis run.  
- **inputs:** The dictionary of identified files.  
- **destination:** The session where the results should be saved.  
- **tags:** Adds metadata tags, such as 'BIDS', for organization.  
- **config:** Includes parameters like target_template and a prefix for the analysis.  
- **job_list.append(job_id):** Adds the job ID to a list to track all submitted jobs.  

**print("Submitting Job: Check Jobs Log", dest.label):** Prints confirmation that the job was submitted.  
**except Block:** If an error occurs, prints a warning message indicating the job couldn’t be sent.  

## Utility Gear Function

In [ ]:
def run_utility_gear(gear_name, subjects, include_patterns, exclude_patterns, filetype, config_input_key,destination):
    
    """Submits a job with specified gear and inputs.

    Args:
        gear_name (string): A Flywheel Gear name.
        subjects (list): list of subjects to run the gear on
        include_patterns (list): patterns to look for in a filename in order to include it in the analysis
        exclude_patterns (list): patterns to look for in a filename to be EXCLUDED from the analysis
        filetype (string): filetypes of interest for this analysis (nifti, dicom, etc)
        config_input_key (string): the key specified in the manifest of the gear to specify inputs 
        destination (string): where to to store the output (session or acquisition) 


    Returns: 
        The function prints out the session for which the job was submitted. This can be verified using the 'Job Log' on the flywheel interface
        
    """
        
    # Initialize gear_job_list
    job_list =  list()
    
    gear =  fw.lookup(f'gears/{gear_name}')

    # Loop through all the subjects in the project
    for session in subjects: 
        #subject = fw.subjects.find_first(f"label={s}")
        session = session.reload()
        # for session in subject.sessions.iter():
        #     session = session.reload()
        print("Parsing... ", session.label, session.subject.label)

        inputs = {}
        

        # Look at every acquisition in the session
        for acquisition in session.acquisitions():
            acquisition = acquisition.reload()
            for file in acquisition.files:
                #print(f'{len(acquisition.files)} found')
                # We only want anatomical Nifti's
                if file.type == filetype and  any(x in file.name for x in include_patterns):
                    if all(pattern not in file.name for pattern in exclude_patterns):
                        inputs[config_input_key] = file
                        print("inputs: ", file.name)
                        try:
                            dest = None
                            # Specify the destination for this analysis
                            if destination == 'session':
                                dest = session
                            elif destination == 'acquisition':
                                dest = acquisition
                            else:
                                print('Invalid destination specified')
                                
                            time_fmt = '%d-%m-%Y_%H-%M-%S'
                            
                            job_id = gear.run(
                                inputs=inputs,
                                destination=dest,
                                tags=['analysis']
                                # ,
                                # config={
                                #     "target_template": target_template,
                                #     "prefix": analysis_tag,
                                # }
                            )
                            job_list.append(job_id)
                            print("Submitting Job: Check Jobs Log", dest.label)
                        except Exception as e:
                            print(f"WARNING: Job cannot be sent for {dest.label}. Error: {e}")
                            
                        break
                        
                                

            # Set to 'None' in order to run axi registration by default
            # target_template = 'None'

            
    

In [ ]:
len(sessions)

In [ ]:
gears_input_config = {'dcm2niix':'dcm2niix_input' , 'dicom-mr-classifier':'dicom'} #@decveloppers- add utility gears as needed?

GEAR_NAME = 'mrr'
project=fw_project
SUBJECTS = project.subjects() #If you need to run this on ALL subjects in the project, pass the following value to your subject variable: [s.label for s in project.subjects.iter()]
INCLUDE_PATTERNS = ['T2','PSIF','CALIPR'] #Add patterns as needed
EXCLUDE_PATTERNS = ['Segmentation', 'Mapping']
FILETYPE = 'nifti'

#CONFIG_INPUT_KEY = gears_input_config[GEAR_NAME]
DESTINATION = 'session'
run_utility_gear(GEAR_NAME, sessions, INCLUDE_PATTERNS, EXCLUDE_PATTERNS, FILETYPE, {}, DESTINATION)


## Analysis Gear Function

In [ ]:
def run_analysis_gear(gear_name, subjects, include_patterns, exclude_patterns, plane_types, filetype, analysis_tag, input_file="", project_level=False,project=None,segtool="recon-all",config={}):
    
    """Submits a job with specified gear and inputs.
    
    Args:
        gear_name (string): A Flywheel Gear name.
        subjects (list): list of subjects to run the gear on
        include_patterns (list): patterns to look for in a filename in order to include it in the analysis
        exclude_patterns (list): patterns to look for in a filename to be EXCLUDED from the analysis
        plane_types (list): planes to include in the analysis (AXI, SAG, COR)
        filetype (string): filetypes of interest for this analysis (nifti, dicom, etc)
        analysis_tag (string): metadata tags for the analysis, such as 'BIDS' for organization.
        
    Returns: 
        The function prints out the session for which the job was submitted. This can be verified using the 'Job Log' on the flywheel interface
        
    """
    # Initialize gear_job_list
    job_list = list()
    gear =  fw.lookup(f'gears/{gear_name}')
    

    # Loop through all the subjects in the project
    for subject in subjects: 
        # subject = fw.subjects.find_first(f"label={s}")
        subject = subject.reload()
        if not (subject.label.startswith('137-')):
            for session in subject.sessions.iter():
                session = session.reload()
                print("Parsing... ", subject.label, session.label)

                inputs = {}
                

                # Look at every acquisition in the session
                for acquisition in session.acquisitions.iter():
                    acquisition = acquisition.reload()
                    for file in acquisition.files:
                        #print(f'{len(acquisition.files)} found')
                        # We only want anatomical Nifti's
                        filename = file.name.upper()
                        
                        if file.type == filetype and any(x in filename for x in include_patterns):
                            print(filename,file.type )
                            if all(pattern not in filename for pattern in exclude_patterns):
                                for plane in plane_types:
                                    if plane in filename:
                                        input_label = plane.lower()
                                        inputs[input_label] = file
                                        print("inputs: ", file.name)
                                        break

                # Set to 'None' in order to run axi registration by default
                target_template = 'None'
                if inputs:
                    try:
                        # The destination for this analysis will be on the session
                        dest = session
                        time_fmt = '%d-%m-%Y_%H-%M-%S'
                        analysis_label = f'{analysis_tag}_{datetime.now().strftime(time_fmt)}'
                        job_id = gear.run(
                            analysis_label=analysis_label,
                            inputs=inputs,
                            destination=dest,
                            tags=['priority', 'analysis','gearname'],
                            config={
                                "age": None
                                # "target_template": target_template,
                                # "prefix": analysis_tag,
                            }
                        )
                        job_list.append(job_id)
                        print("Submitting Job: Check Jobs Log", dest.label)
                    except Exception as e:
                        print(f"WARNING: Job cannot be sent for {dest.label}. Error: {e}")

        

In [ ]:
## How to use the run_analysis_gear function

GEAR_NAME = 'recon-all'
SUBJECTS = project.subjects() #If you need to run this on ALL subjects in the project, pass the following value to your subject variable: [s.label for s in project.subjects.iter()]
INCLUDE_PATTERNS = 'T2' #Add patterns as needed
EXCLUDE_PATTERNS = ['Segmentation', 'Align', 'Mapping']
PLANE_TYPES = ['AXI'] #, 'COR', 'SAG'
FILETYPE = 'nifti'
ANALYSIS_TAG =  'gpu'

run_analysis_gear(GEAR_NAME, SUBJECTS, INCLUDE_PATTERNS, EXCLUDE_PATTERNS, PLANE_TYPES, FILETYPE, ANALYSIS_TAG)

### Running Project-level gears: Example - Volumetrics gear 

In [ ]:
GEAR_NAME = 'volumetrics'
FILETYPE = 'csv'
ANALYSIS_TAG =  'analysis, volumetrics'
SUBJECTS = []
INCLUDE_PATTERNS = '' 
EXCLUDE_PATTERNS = []
PLANE_TYPES = []

group_name = "global_map" 
group = fw.lookup(group_name)
projects = group.projects()

for project in projects[18:]:
  print(project.label)
  fw_project = fw.projects.find_first(f"label={project.label}")
  fw_project = fw_project.reload()
  config= {
      #You can either input age_min and age_max in MONTHS, or choose an option for age_range from one of the following:
      # "Infants (0-12 months)",
      # "1st 1000 Days (0-32 months)",
      # "Toddlers (1-3 years)",
      # "Preschool (3-6 years)",
      # "School-age Children (6-12 years)",
      # "Adolescents (12-18 years)",
      # "Young Adults (18-34 years)",
      # "Adults (35-89 years)",
      # "All Ages (0-100 years)"
    "age_range": "",
    "threshold": 1.5,
    "age_min_months": 1,
    "age_max_months": 3,
    "age_unit":"months"
  }


  run_analysis_gear(GEAR_NAME, SUBJECTS, INCLUDE_PATTERNS, EXCLUDE_PATTERNS, PLANE_TYPES, FILETYPE, ANALYSIS_TAG, input_file="", project_level=True, project=fw_project, segtool="gearname", config={})

In [ ]:
subjects =[]
sessions = []
for session in project.sessions():
    session = session.reload()
    recon_asys = [asys for asys in session.analyses if asys.gear_info is not None and asys.gear_info.get('name') == "recon-all-clinical" and asys.job.get('state') == 'complete' and 'gambas' not in asys.label]
    mrr_asys  = [asys for asys in session.analyses if asys.gear_info is not None and asys.gear_info.get('name') == "mrr" and asys.job.get('state') == 'complete']
    if not recon_asys and mrr_asys and "137-" not in session.subject.label:
        print(session.label)
        sessions.append(session)
        

In [ ]:
def run_segmentation(project, gearname):
    gear =  fw.lookup(f'gears/{gearname}')
    
    # Initialize gear_job_list
    job_list = list()

    for subject in project.subjects():
        subject = subject.reload()
        for session in subject.sessions():
            analysis_tag = gearname
            session = session.reload()
            inputfile = None
            print("Parsing... ", subject.label, session.label)

            inputs = {}

            analyses = session.analyses

            # If there are no analyses containers, we know that this gear was not run
            if len(analyses) == 0:
                run = 'False'
                status = 'NA'
                print('No analysis containers')
            else:
                try:

                    mrr_matches = [asys for asys in analyses if asys.gear_info is not None and asys.gear_info.get('name') == "mrr" and asys.job.get('state') == 'complete']
                    gambas_matches =  [asys for asys in analyses if "gambas" in asys.label]

                    for matches in [mrr_matches]:
                        # If there are no matches, the gear didn't run
                        if len(matches) == 0:
                            run = 'False'
                            status = 'NA'
                        # If there is one match, that's our target
                        elif len(matches) == 1:
                            run = 'True'
                            #status = matches[0].job.get('state')
                            print(status)

                            for file in matches[0].files:  
                                if re.search(r'mrr.*\.nii.gz', file.name):
                                    inputfile = file
                                    print(inputfile.name)
                                    analysis_tag = f'{gearname}-mrr-axireg'
                                elif re.search(r'ResCNN.*\.nii.gz', file.name):
                                    inputfile = file
                                    print(inputfile.name)
                                    analysis_tag = f'{gearname}-gambas'
                                    
                        else:
                                #consider gambas as well as recon-all
                                last_run_date = max([asys.created for asys in matches])
                                last_run_analysis = [asys for asys in matches if asys.created == last_run_date]

                                # There should only be one exact match
                                last_run_analysis = last_run_analysis[0]

                                run = 'True'
                                #status = last_run_analysis.job.get('state')
                                for file in matches[0].files:  
                                    if re.search(r'mrr.*\.nii.gz', file.name):
                                        inputfile = file
                                        print(inputfile.name)
                                        analysis_tag = f'{gearname}-mrr-axireg'
                                    elif re.search(r'ResCNN.*\.nii.gz', file.name):
                                        inputfile = file
                                        print(inputfile.name)
                                        analysis_tag = f'{gearname}-gambas'

                        if inputfile:
                            inputs["input"]= inputfile
                            print("Input file" , inputfile.name)

                            try:
                                # The destination for this analysis will be on the session
                                target_template = 'None'
                                dest = session
                                time_fmt = '%d-%m-%Y_%H-%M-%S'

                                analysis_label = f'{analysis_tag}_{datetime.now().strftime(time_fmt)}'
                                job_id = gear.run(
                                    analysis_label=analysis_label,
                                    inputs=inputs,
                                    destination=dest,
                                    tags=['priority', 'analysis',gearname],
                                    config={
                                        # "target_template": target_template,
                                        "age": "None"
                                    }
                                )
                                job_list.append(job_id)
                                print("Submitting Job: Check Jobs Log", dest.label)
                            except Exception as e:
                                print(f"WARNING: Job cannot be sent for {dest.label}. Error: {e}")

                except Exception as e:
                        print(f"Exception caught for {subject.label}: ", e)

In [ ]:
fw_project.label

In [ ]:
run_segmentation(fw_project, 'recon-all-clinical')

In [ ]:
import os
from datetime import datetime
import pytz
from IPython.display import display
import re
import pandas as pd
import math


from dotenv import load_dotenv
import flywheel
# Connect to your Flywheel instance

fw = flywheel.Client(api_key=api_key)
display(f"User: {fw.get_current_user().firstname} {fw.get_current_user().lastname}")



def run_circumference_gear(project, session=None):
    gear =  fw.lookup('gears/circumference')
    
    # Initialize gear_job_list
    job_list = list()
    analysis_tag = 'circumference'

    if session is not None:

    # for subject in project.subjects():
    #     subject = subject.reload()
    #     for session in subject.sessions():
            
    #         session = session.reload()
        inputfile = None
        print("Parsing... ", session.label)

        inputs = {}

        analyses = session.analyses

        # If there are no analyses containers, we know that this gear was not run
        if len(analyses) == 0:
            run = 'False'
            status = 'NA'
            print('No analysis containers')
        else:
            try:

                mrr_matches = [asys for asys in analyses if asys.gear_info is not None and asys.gear_info.get('name') == "mrr" and asys.job.get('state') == 'complete']
                gambas_matches =  [asys for asys in analyses if "gambas" in asys.label]

                for matches in [gambas_matches]:
                    print(f'Found {len(matches)}')
                    # If there are no matches, the gear didn't run
                    if len(matches) == 0:
                        run = 'False'
                        status = 'NA'
                    # If there is one match, that's our target
                    elif len(matches) == 1:
                        run = 'True'
                        #status = matches[0].job.get('state')
                        print(status)

                        for file in matches[0].files:  
                            if re.search('mrr.nii.gz', file.name):
                                inputfile = file
                                print(inputfile.name)
                                analysis_tag = 'circumference-mrr-axireg'
                            elif re.search('ResCNN.nii.gz', file.name):
                                inputfile = file
                                print(inputfile.name)
                                analysis_tag = 'circumference-gambas'
                            elif re.search('T2w_gambas.nii.gz', file.name):
                                inputfile = file
                                print(inputfile.name)
                                analysis_tag = 'circumference-gambas'
                                
                    else:
                            #consider gambas as well as recon-all
                            last_run_date = max([asys.created for asys in matches])
                            last_run_analysis = [asys for asys in matches if asys.created == last_run_date]

                            # There should only be one exact match
                            last_run_analysis = last_run_analysis[0]

                            run = 'True'
                            #status = last_run_analysis.job.get('state')
                            for file in matches[0].files:  
                                if re.search('mrr.nii.gz', file.name):
                                    inputfile = file
                                    print(inputfile.name)
                                    analysis_tag = 'circumference-mrr-axireg'
                                elif re.search('ResCNN.nii.gz', file.name):
                                    inputfile = file
                                    print(inputfile.name)
                                    analysis_tag = 'circumference-gambas'

                                elif re.search('T2w_gambas.nii.gz', file.name):
                                    inputfile = file
                                    print(inputfile.name)
                                    analysis_tag = 'circumference-gambas'
                                

                    if inputfile:
                        inputs["input"]= inputfile
                        print("Input file" , inputfile.name)

                        try:
                            # The destination for this analysis will be on the session
                            target_template = 'None'
                            dest = session
                            time_fmt = '%d-%m-%Y_%H-%M-%S'

                            analysis_label = f'{analysis_tag}_{datetime.now().strftime(time_fmt)}'
                            job_id = gear.run(
                                analysis_label=analysis_label,
                                inputs=inputs,
                                destination=dest,
                                tags=['batch','analysis','circumference'],
                                config={
                                    # "target_template": target_template,
                                    "prefix": analysis_tag
                                }
                            )
                            job_list.append(job_id)
                            print("Submitting Job: Check Jobs Log", dest.label)
                        except Exception as e:
                            print(f"WARNING: Job cannot be sent for {dest.label}. Error: {e}")

            except Exception as e:
                    print(f"Exception caught for {session.label}: ", e)


In [ ]:
def run_form_parser(project):
    gear =  fw.lookup('gears/form-parser')
    analysis_tag = "form-parser"

    # Initialize gear_job_list
    job_list = list()

    try:
        # The destination for this analysis will be on the project
        print("Sending job ...", project.label)
       
        dest = project
        time_fmt = '%d-%m-%Y_%H-%M-%S'
        analysis_label = f'{analysis_tag}_{datetime.now().strftime(time_fmt)}'
        job_id = gear.run(
            analysis_label=analysis_label,
            destination=dest,
            tags=['analysis, form-parser','qc','batch']
            
        )
        job_list.append(job_id)
        print("Submitting Job: Check Jobs Log", dest.label)
    except Exception as e:
        print(f"WARNING: Job cannot be sent for {dest.label}. Error: {e}")


In [ ]:
group_names = ["global_map"]
project_ids = []
for group_name in group_names:    
    # Get the group
    group = fw.lookup(f"{group_name}")
    group = group.reload()
    # Get the projects in the group
    projects = group.projects()
    project_ids.extend([project.label for project in projects])

print(f"Found {len(project_ids)} projects.")


In [ ]:
for project_label in project_ids:
    fw_project = fw.projects.find_first(f'label={project_label}')
    fw_project = fw_project.reload()
    run_form_parser(fw_project)
   

In [ ]:
def run_volumetrics(project,segtool, input_file=""):
    ##RUNNING VOLUMETRICS GEAR
    volumes = None
    job_list = list()
    gear =  fw.lookup('gears/orbit')
    analysis_tag = "orbit"
    if project is not None:
        if input_file == "":
            print("Looking for input file")
            vols = [file for file in project.files if segtool in file.name and "volume" in file.name and file.name.endswith('.csv')]
            #get the file by last_created date
            vols = sorted(vols, key=lambda x: x.created, reverse=True)
            if vols:
                volumes = vols[0]
        else:
            volumes = input_file

        if volumes:
            inputs = {"input": volumes}
            
            download_dir = os.path.join("/Users/Hajer/unity/fw-notebooks/Data")
            fileName = volumes.name
            download_path = os.path.join(download_dir, fileName)
            

            # Download the file
            print('downloading file', download_path)
            volumes.download(download_path)
            df = pd.read_csv(download_path)
            

            # clean_age = pd.to_numeric(
            # df['template_age'].str.replace("M", "", regex=False),
            # errors='coerce'
            # )
            print(df.columns)
            df.rename(columns={"dicom_age_in_months": "age"}, errors="ignore",inplace=True)
            df["age"] = df["age"].str.replace("M", "", regex=False)

            clean_age = pd.to_numeric(
            df['age'])
            # Drop NaNs, floor the rest
            floored = clean_age.dropna().apply(math.floor)

            # Get the min
            min_age = int(floored.min())
            max_age = int(floored.max())

            config= {
        
                "age_range": "",
                "threshold": 1.5,
                "age_min_months": min_age,
                "age_max_months": max_age,
                "age_unit":"months"
            }

            try:
                # The destination for this analysis will be on the project
                print("Sending job ...")
                print("Config" , config)
                dest = project
                time_fmt = '%d-%m-%Y_%H-%M-%S'
                analysis_label = f'{analysis_tag}_{datetime.now().strftime(time_fmt)}'
                job_id = gear.run(
                    analysis_label=analysis_label,
                    inputs=inputs,
                    destination=dest,
                    tags=['analysis, volumetric'],
                    config=config
                )
                #job_list.append(job_id)
                print("Submitting Job: Check Jobs Log", dest.label)
            except Exception as e:
                print(f"WARNING: Job cannot be sent for {dest.label}. Error: {e}")
        else:
            print(f"Skipping this project ... no {segtool} file found.")

In [ ]:
project_ids

In [ ]:

for project_label in project_ids:
    fw_project = fw.projects.find_first(f'label={project_label}')
    fw_project = fw_project.reload()
    # run_form_parser(fw_project)

    run_volumetrics(fw_project,"recon-all", input_file="")



In [ ]:
def run_custom_info_sync(project, input_file=""):
    ##RUNNING custom_info_sync GEAR
    
    job_list = list()
    gear =  fw.lookup('gears/custom-information-sync')
    analysis_tag = "custom-information-sync"
    config={"dry_run": False}

    try:
        # The destination for this analysis will be on the project
        print("Sending job ...")
        print("Config" , config)
        dest = project
        time_fmt = '%d-%m-%Y_%H-%M-%S'
        analysis_label = f'{analysis_tag}_batch_{datetime.now().strftime(time_fmt)}'
        job_id = gear.run(
            analysis_label=analysis_label,
            destination=dest,
            tags=['analysis', analysis_tag],
            config=config
        )
        job_list.append(job_id)
        print("Submitting Job: Check Jobs Log", dest.label)
    except Exception as e:
        print(f"WARNING: Job cannot be sent for {dest.label}. Error: {e}")
       

In [ ]:
group_names = ["prisma","global_map"] #prisma #global_map
project_ids = []
for group_name in group_names:    
    # Get the group
    group = fw.lookup(f"{group_name}")
    group = group.reload()
    # Get the projects in the group
    projects = group.projects()
    project_ids.extend([project.label for project in projects])

print(f"Found {len(project_ids)} projects.")


In [ ]:
for project_label in project_ids:
    fw_project = fw.projects.find_first(f'label={project_label}')
    fw_project = fw_project.reload()
    # run_form_parser(fw_project)
    run_custom_info_sync(fw_project,input_file="")



#### PULL CUSTOM-INFO-SYNC TO RERUN GEAR WHERE VALUE IS MISSING

In [ ]:
from pathlib import Path

for project_label in project_ids:

    project = fw.projects.find_first(f'label={project_label}')
    print(project.label)
    project = project.reload()

    analyses = project.analyses

    last_run_date_custom = None
    all_runs = []
    run_states = []
    custom_info_sync = None
    for asys in analyses:

        try:

            all_runs.append(asys.reload().gear_info.get('name'))
            run_states.append(asys.reload().job.get('state'))

            gear_name = asys.gear_info.get('name')
            created_date = asys.created

            if gear_name == "custom-information-sync" and "batch" in asys.label:
                last_run_date_custom = max(last_run_date_custom, created_date) if last_run_date_custom else created_date
                custom_info_sync = asys

        
        except Exception as e:
            print(f'Exception caught:  {e}')

    last_run_date_custom = last_run_date_custom.strftime('%Y-%m-%d') if last_run_date_custom else None
    print("Date custom_info_sync last ran: ", last_run_date_custom)


    # Download all the custom-info-sync files
    for f in custom_info_sync.files:
        if f.name.endswith('session-information.csv'):
            custom_info_file = f
            download_dir = Path("/Users/Hajer/unity/GF Sprint25/August_London/session-information")
            download_dir.mkdir(parents=True,exist_ok=True)
            download_path = os.path.join(download_dir , custom_info_file.name)
            custom_info_file.download(download_path)
            break



In [ ]:
directory="/Users/Hajer/unity/GF Sprint25/August_London/session-information"
dfs = []
#walk the directory, read every csv, and concatenate to one dataframe
for (root,dirs,files) in os.walk(directory):
    
    for file in files:
        if file.endswith('.csv'):
            print(f"Processing file: {file}")
            df = pd.read_csv(os.path.join(directory,file))
            dfs.append(df)

df_demo = pd.concat(dfs, ignore_index=True)

df_demo.to_csv("/Users/Hajer/unity/GF Sprint25/August_London/session-information/ALL_SESSIONINFO_UNITY.csv")

In [ ]:
df_demo.project_id.value_counts()

In [ ]:
missing_hc_df = df_demo[df_demo['childTimepointHC_MRI_cm'].isna()]#
print(missing_hc_df.project_id.value_counts())
missing_hc_df[["group_id",	"project_id",	"subject_id",	"session_id",'estimated_hc_cm','childTimepointHC_MRI_cm']]



In [ ]:
for _, row in missing_hc_df.iterrows():
    project = fw.projects.find_first(f'label={row["project_id"]}')
    print(project.label)
    project = project.reload()
    session = project.sessions.find_first(f'label="{row["session_id"]}"')
    
    session = session.reload()

    # subject = project.subjects.find_first(f'label="{row["subject_id"]}"')
    # print(row["subject_id"])
    # subject = subject.reload()
    run_circumference_gear(project,session)

In [ ]:
for session in fw_project.sessions():
    session = session.reload()
    run_circumference_gear(project,session)

In [ ]:
rows = []

for _, row in missing_hc_df.iterrows():
    project = fw.projects.find_first(f'label={row["project_id"]}')
    print(project.label)
    project = project.reload()
    session = project.sessions.find_first(f'label="{row["session_id"]}"')
    
    session = session.reload()
    
    analyses = session.analyses  # or session.reload().analyses if needed

    # Find matches
    mrr_matches = [
        asys for asys in analyses
        if asys.gear_info is not None
        and asys.gear_info.get('name') == "mrr"
        and asys.job.get('state') == 'complete'
    ]

    gambas_matches = [
        asys for asys in analyses
        if "gambas" in asys.label.lower()  # lower() in case of case mismatch
    ]

    # Only keep sessions with no mrr or gambas analyses
    if len(mrr_matches) == 0 and len(gambas_matches) == 0:
        rows.append({
            'project': project.label,
            'subject':row["subject_id"],
            'session': row["session_id"]
        })

# Create DataFrame
df_missing_analyses = pd.DataFrame(rows)

In [ ]:
df_missing_analyses[['project','subject','session']].to_csv(os.path.join(directory,"missing_gambas_mrr.csv"))

In [ ]:
import flywheel
import os
from datetime import datetime
 
def run_minimorph_jobs(fw):
    """
    Run minimorph jobs on the most recent gambas analysis for each session
    if minimorph hasn't already been completed.
    """
    
    # Initialize Flywheel client
    # api_key = os.environ.get('FW_CLI_API_KEY')
    # fw = flywheel.Client(api_key=api_key)
    
    # Configuration
    group_name = 'prisma' #"global_map" , "prisma"
    group = fw.lookup(group_name)
    gear = fw.lookup('gears/minimorph')
    gear_version = '1.0.20'
    project_list = [ 'Malawi-Khula-Hyperfine']  
    # 'UCT-SPACE_EXPLORE-Hyperfine', 'Ethiopia-BCD-Hyperfine',
 
    # Initialize job tracking
    job_list = []
    processed_sessions = 0
    skipped_sessions = 0
    failed_sessions = 0
    
    print(f"🚀 Starting minimorph job submission for gear version {gear_version}")
    print(f"📋 Processing projects: {project_list}")
    
    # Get all projects in the group
    projects = fw.projects()
    
    # Loop over specified projects
    for project in projects:
        if project.label not in project_list:
            continue
            
        print(f"\n📁 Processing project: {project.label}")
        
        # Loop through subjects and sessions
        for subject in project.subjects.iter():
            for session in subject.sessions.iter():
                session = session.reload()
                session_id = f"{project.label}/{subject.label}/{session.label}"
                print(f"\n🔍 Checking session: {session_id}")
                
                try:
                    # Check if minimorph already completed specifically for gambas input
                    if has_completed_minimorph_gambas(session, gear_version):
                        print(f"✅ minimorph {gear_version} with gambas input already complete, skipping")
                        skipped_sessions += 1
                        continue
                    
                    # Find the most recent gambas analysis
                    gambas_file = find_latest_gambas_file(session)
                    if not gambas_file:
                        print(f"⚠️ No suitable gambas file found, skipping")
                        failed_sessions += 1
                        continue
                    
                    print(f"✅ Found gambas file: {gambas_file.name}")
                    
                    # Submit minimorph job
                    job_id = submit_minimorph_job(gear, session, gambas_file, gear_version)
                    job_list.append(job_id)
                    processed_sessions += 1
                    print(f"🚀 Submitted minimorph job (ID: {job_id})")
                    
                except Exception as e:
                    print(f"❌ Error processing session {session_id}: {str(e)}")
                    failed_sessions += 1
                    continue
    
    # Summary
    print(f"\n📊 Summary:")
    print(f"   ✅ Jobs submitted: {processed_sessions}")
    print(f"   ⏭️ Sessions skipped: {skipped_sessions}")
    print(f"   ❌ Sessions failed: {failed_sessions}")
    print(f"   📋 Total job IDs: {len(job_list)}")
    
    return job_list
 
def has_completed_minimorph_gambas(session, target_version):
    """
    Check if session already has a completed minimorph analysis of the target version specifically for gambas input.
    """
    for analysis in session.analyses:
        if not analysis.gear_info:
            continue
            
        gear_name = analysis.gear_info.get('name', '').lower()
        gear_version = analysis.gear_info.get('version', '')
        
        if 'minimorph' in gear_name and gear_version == target_version:
            # Check if this minimorph analysis was run with gambas input
            if 'gambas' in analysis.label.lower():
                job_state = analysis.job.get('state') if analysis.job else None
                if job_state == 'complete':
                    return True
    return False
 
def has_completed_minimorph(session, target_version):
    """
    Check if session already has a completed minimorph analysis of the target version.
    """
    for analysis in session.analyses:
        if not analysis.gear_info:
            continue
            
        gear_name = analysis.gear_info.get('name', '').lower()
        gear_version = analysis.gear_info.get('version', '')
        
        if 'minimorph' in gear_name and gear_version == target_version:
            job_state = analysis.job.get('state') if analysis.job else None
            if job_state == 'complete':
                return True
    return False
 
def find_latest_gambas_file(session):
    """
    Find the most recent gambas analysis file in the session or its acquisitions.
    Returns the first suitable gambas file ending with 'rec-axi_T2w_gambas.nii.gz' from the latest gambas analysis, or None if not found.
    """
    print(f"   Checking {len(session.analyses)} analyses in session")
    
    # Debug: Print all session analyses
    for i, analysis in enumerate(session.analyses):
        gear_name = analysis.gear_info.get('name', 'No gear name') if analysis.gear_info else 'No gear_info'
        print(f"   Session Analysis {i}: label='{analysis.label}', gear='{gear_name}'")
    
    # Find all gambas analyses in this session
    gambas_analyses = []
    for analysis in session.analyses:
        if is_gambas_analysis(analysis):
            gambas_analyses.append(analysis)
            print(f"   Found gambas in session: {analysis.label}")
    
    # Also check acquisitions for gambas analyses
    print(f"   Checking acquisitions for gambas analyses...")
    for acquisition in session.acquisitions():
        acquisition = acquisition.reload()
        print(f"   Acquisition: {acquisition.label} has {len(acquisition.analyses)} analyses")
        
        for i, analysis in enumerate(acquisition.analyses):
            gear_name = analysis.gear_info.get('name', 'No gear name') if analysis.gear_info else 'No gear_info'
            print(f"     Acquisition Analysis {i}: label='{analysis.label}', gear='{gear_name}'")
            
            if is_gambas_analysis(analysis):
                gambas_analyses.append(analysis)
                print(f"   Found gambas in acquisition: {analysis.label}")
    
    if not gambas_analyses:
        print("   No gambas analyses found in session or acquisitions")
        return None
    
    print(f"   Found {len(gambas_analyses)} gambas analysis(es) total")
    
    # Use the latest gambas analysis (assuming they're ordered chronologically)
    latest_gambas = gambas_analyses[-1]
    print(f"   Using latest gambas analysis: {latest_gambas.label}")
    
    # Debug: Print all files in the analysis
    print(f"   Files in analysis: {[f.name for f in latest_gambas.files]}")
    
    # Find gambas output files - specifically look for files ending with "rec-axi_T2w_gambas.nii.gz"
    pattern = re.compile(r"rec-axi(?:_run-\d+)?_T2w_gambas\.nii\.gz$")

    gambas_files = [
        f for f in latest_gambas.files
        if pattern.search(f.name)
    ]

    if not gambas_files:
        print(f"   No files ending with 'rec-axi(_run-XX)_T2w_gambas.nii.gz' found in analysis {latest_gambas.label}")
        return None
    
    print(f"   Found {len(gambas_files)} gambas file(s): {[f.name for f in gambas_files]}")
    
    # Return the first matching file
    return gambas_files[0]
 
def is_gambas_analysis(analysis):
    """
    Check if an analysis is a gambas analysis by checking gear name or analysis label.
    """
    # Check gear name - must be exactly 'gambas' gear
    if analysis.gear_info and analysis.gear_info.get('name'):
        gear_name = analysis.gear_info.get('name').lower()
        if gear_name == 'gambas':
            return True
    
    # If no gear_info, check the analysis label for gambas version pattern
    if analysis.label:
        label = analysis.label.lower()
        # Look for patterns like 'gambas/0.4.14' or 'gambas/0.4.17'
        if 'gambas/0.4.14' in label or 'gambas/0.4.17' in label:
            return True
    
    return False
 
def submit_minimorph_job(gear, session, gambas_file, gear_version):
    """
    Submit a minimorph job for the given session and gambas file.
    """
    inputs = {'input': gambas_file}
    
    # Create a unique analysis label with timestamp and gambas identifier
    timestamp = datetime.now().strftime("%d-%m-%Y_%H-%M-%S")
    analysis_label = f'minimorph_gambas_{gear_version}_{timestamp}'
    
    # Submit the job
    job_id = gear.run(
        analysis_label=analysis_label,
        inputs=inputs,
        destination=session,
        tags=[],
        config={}
    )
    
    return job_id
 
def check_job_status(fw, job_ids):
    """
    Check the status of submitted jobs.
    """
    print(f"\n🔍 Checking status of {len(job_ids)} jobs:")
    
    status_counts = {}
    for job_id in job_ids:
        try:
            job = fw.get_job(job_id)
            state = job.state
            status_counts[state] = status_counts.get(state, 0) + 1
            print(f"   Job {job_id}: {state}")
        except Exception as e:
            print(f"   Job {job_id}: Error - {str(e)}")
            status_counts['error'] = status_counts.get('error', 0) + 1
    
    print(f"\n📊 Job Status Summary:")
    for state, count in status_counts.items():
        print(f"   {state}: {count}")
 
if __name__ == "__main__":
    # Run the main function
    job_list = run_minimorph_jobs(fw)
    
    # Optionally check job status after submission
    if job_list:
        print(f"\n⏳ Waiting a moment before checking job status...")
        import time
        time.sleep(5)
        
        # api_key = os.environ.get('FW_CLI_API_KEY')
        # fw = flywheel.Client(api_key=api_key)
        check_job_status(fw, job_list)

### Run recon-all 

In [ ]:
import flywheel
import os
from datetime import datetime
 
def run_recon_all_jobs(fw):
    """
    Run minimorph jobs on the most recent gambas analysis for each session
    if minimorph hasn't already been completed.
    """
    
    # Initialize Flywheel client
    # api_key = os.environ.get('FW_CLI_API_KEY')
    # fw = flywheel.Client(api_key=api_key)
    
    # Configuration
    group_name = 'global_map' #"global_map" , "prisma"
    group = fw.lookup(group_name)
    gear = fw.lookup('gears/recon-all-clinical')
    gear_version = '0.4.7'
    project_list = [ 'Botswana-MOTHEO']  
    # 'UCT-SPACE_EXPLORE-Hyperfine', 'Ethiopia-BCD-Hyperfine',
 
    # Initialize job tracking
    job_list = []
    processed_sessions = 0
    skipped_sessions = 0
    failed_sessions = 0
    
    print(f"🚀 Starting recon-all-clinical job submission for gear version {gear_version}")
    print(f"📋 Processing projects: {project_list}")
    
    # Get all projects in the group
    projects = fw.projects()
    
    # Loop over specified projects
    for project in projects:
        if project.label not in project_list:
            continue
            
        print(f"\n📁 Processing project: {project.label}")
        
        # Loop through subjects and sessions
        for subject in project.subjects.iter():
            if not (subject.label.startswith("137-")):
                for session in subject.sessions.iter():
                    session = session.reload()
                    session_id = f"{project.label}/{subject.label}/{session.label}"
                    print(f"\n🔍 Checking session: {session_id}")
                    
                    try:
                        # Check if minimorph already completed specifically for gambas input
                        if has_completed_recon_all_gambas(session, gear_version):
                            print(f"✅ recon-all {gear_version} with gambas input already complete, skipping")
                            skipped_sessions += 1
                            continue
                        
                        # Find the most recent gambas analysis
                        gambas_file = find_latest_gambas_file(session)
                        if not gambas_file:
                            print(f"⚠️ No suitable gambas file found, skipping")
                            failed_sessions += 1
                            continue
                        
                        print(f"✅ Found gambas file: {gambas_file.name}")
                        
                        # Submit minimorph job
                        job_id = submit_recon_all_job(gear, session, gambas_file, gear_version)
                        job_list.append(job_id)
                        processed_sessions += 1
                        print(f"🚀 Submitted recon-all job (ID: {job_id})")
                        
                    except Exception as e:
                        print(f"❌ Error processing session {session_id}: {str(e)}")
                        failed_sessions += 1
                        continue
        
    # Summary
    print(f"\n📊 Summary:")
    print(f"   ✅ Jobs submitted: {processed_sessions}")
    print(f"   ⏭️ Sessions skipped: {skipped_sessions}")
    print(f"   ❌ Sessions failed: {failed_sessions}")
    print(f"   📋 Total job IDs: {len(job_list)}")
    
    return job_list
 
def has_completed_recon_all_gambas(session, target_version):
    """
    Check if session already has a completed minimorph analysis of the target version specifically for gambas input.
    """
    for analysis in session.analyses:
        if not analysis.gear_info:
            continue
            
        gear_name = analysis.gear_info.get('name', '').lower()
        gear_version = analysis.gear_info.get('version', '')
        
        if 'recon-all-clinical' in gear_name and gear_version == target_version:
            # Check if this minimorph analysis was run with gambas input
            if 'gambas' in analysis.label.lower():
                job_state = analysis.job.get('state') if analysis.job else None
                if job_state == 'complete':
                    return True
    return False
 
def has_completed_recon_all(session, target_version):
    """
    Check if session already has a completed recon-all analysis of the target version.
    """
    for analysis in session.analyses:
        if not analysis.gear_info:
            continue
            
        gear_name = analysis.gear_info.get('name', '').lower()
        gear_version = analysis.gear_info.get('version', '')
        
        if 'recon-all-clinical' in gear_name and gear_version == target_version:
            job_state = analysis.job.get('state') if analysis.job else None
            if job_state == 'complete':
                return True
    return False
 
def find_latest_gambas_file(session):
    """
    Find the most recent gambas analysis file in the session or its acquisitions.
    Returns the first suitable gambas file ending with 'rec-axi_T2w_gambas.nii.gz' from the latest gambas analysis, or None if not found.
    """
    print(f"   Checking {len(session.analyses)} analyses in session")
    
    # Debug: Print all session analyses
    for i, analysis in enumerate(session.analyses):
        gear_name = analysis.gear_info.get('name', 'No gear name') if analysis.gear_info else 'No gear_info'
        print(f"   Session Analysis {i}: label='{analysis.label}', gear='{gear_name}'")
    
    # Find all gambas analyses in this session
    gambas_analyses = []
    for analysis in session.analyses:
        if is_gambas_analysis(analysis):
            gambas_analyses.append(analysis)
            print(f"   Found gambas in session: {analysis.label}")
    
    # Also check acquisitions for gambas analyses
    print(f"   Checking acquisitions for gambas analyses...")
    for acquisition in session.acquisitions():
        acquisition = acquisition.reload()
        print(f"   Acquisition: {acquisition.label} has {len(acquisition.analyses)} analyses")
        
        for i, analysis in enumerate(acquisition.analyses):
            gear_name = analysis.gear_info.get('name', 'No gear name') if analysis.gear_info else 'No gear_info'
            print(f"     Acquisition Analysis {i}: label='{analysis.label}', gear='{gear_name}'")
            
            if is_gambas_analysis(analysis):
                gambas_analyses.append(analysis)
                print(f"   Found gambas in acquisition: {analysis.label}")
    
    if not gambas_analyses:
        print("   No gambas analyses found in session or acquisitions")
        return None
    
    print(f"   Found {len(gambas_analyses)} gambas analysis(es) total")
    
    # Use the latest gambas analysis (assuming they're ordered chronologically)
    latest_gambas = gambas_analyses[-1]
    print(f"   Using latest gambas analysis: {latest_gambas.label}")
    
    # Debug: Print all files in the analysis
    print(f"   Files in analysis: {[f.name for f in latest_gambas.files]}")
    
    # Find gambas output files - specifically look for files ending with "rec-axi_T2w_gambas.nii.gz"
    pattern = re.compile(r"rec-axi(?:_run-\d+)?_T2w_gambas\.nii\.gz$")

    gambas_files = [
        f for f in latest_gambas.files
        if f.name.endswith("T2w_gambas.nii.gz")
    ]

    if not gambas_files:
        print(f"   No files ending with 'rec-axi(_run-XX)_T2w_gambas.nii.gz' found in analysis {latest_gambas.label}")
        return None
    
    print(f"   Found {len(gambas_files)} gambas file(s): {[f.name for f in gambas_files]}")
    
    # Return the first matching file
    return gambas_files[0]
 
def is_gambas_analysis(analysis):
    """
    Check if an analysis is a gambas analysis by checking gear name or analysis label.
    """
    # Check gear name - must be exactly 'gambas' gear
    if analysis.gear_info and analysis.gear_info.get('name'):
        gear_name = analysis.gear_info.get('name').lower()
        if gear_name == 'gambas':
            return True
    
    # If no gear_info, check the analysis label for gambas version pattern
    if analysis.label:
        label = analysis.label.lower()
        # Look for patterns like 'gambas/0.4.14' or 'gambas/0.4.17'
        if "gambas" in label and ("0.4.17" in label or "0.4.14" in label):
        # if pattern.search(label):
            return True
    
    return False
 
def submit_recon_all_job(gear, session, gambas_file, gear_version):
    """
    Submit a recon-all job for the given session and gambas file.
    """
    inputs = {'input': gambas_file}
    
    # Create a unique analysis label with timestamp and gambas identifier
    timestamp = datetime.now().strftime("%d-%m-%Y_%H-%M-%S")
    analysis_label = f'recon-all-clinical_gambas_{gear_version}_{timestamp}'
    
    # Submit the job
    job_id = gear.run(
        analysis_label=analysis_label,
        inputs=inputs,
        destination=session,
        tags=[],
        config={}
    )
    
    return job_id
 
def check_job_status(fw, job_ids):
    """
    Check the status of submitted jobs.
    """
    print(f"\n🔍 Checking status of {len(job_ids)} jobs:")
    
    status_counts = {}
    for job_id in job_ids:
        try:
            job = fw.get_job(job_id)
            state = job.state
            status_counts[state] = status_counts.get(state, 0) + 1
            print(f"   Job {job_id}: {state}")
        except Exception as e:
            print(f"   Job {job_id}: Error - {str(e)}")
            status_counts['error'] = status_counts.get('error', 0) + 1
    
    print(f"\n📊 Job Status Summary:")
    for state, count in status_counts.items():
        print(f"   {state}: {count}")
 
if __name__ == "__main__":
    # Run the main function
    job_list = run_recon_all_jobs(fw)
    
    # Optionally check job status after submission
    if job_list:
        print(f"\n⏳ Waiting a moment before checking job status...")
        import time
        time.sleep(5)
        
        # api_key = os.environ.get('FW_CLI_API_KEY')
        # fw = flywheel.Client(api_key=api_key)
        check_job_status(fw, job_list)